### Baseline

In [ ]:
# Configuración de recursos según la máquina
entorno = 'oficina'  # cambiar a 'oficina' según corresponda

if entorno == 'casa':
    # i7-12700K, 12 cores, 32GB RAM
    duckdb_threads = 8
    duckdb_memory_limit = '16GB'
    rf_n_jobs = 8
    lgb_threads = 8
elif entorno == 'oficina':
    # 16 cores, 63.8GB RAM
    duckdb_threads = 12
    duckdb_memory_limit = '45GB'
    rf_n_jobs = 10
    lgb_threads = 10          # 8 por precaución: la oficina se apagó con CPU al 100% el 27/09
else:
    raise ValueError("entorno debe ser 'casa' u 'oficina'")

print(f"Entorno: {entorno} | DuckDB: {duckdb_threads} threads, {duckdb_memory_limit} | RF: n_jobs={rf_n_jobs} | LGB: {lgb_threads} threads")

#### Librerías y Path Variables

In [ ]:
# Importando Librerías

import os, sys
os.environ['PATH'] = os.path.join(sys.prefix, 'Library', 'bin') + os.pathsep + os.environ['PATH']
import duckdb

import pandas as pd
import numpy as np
import pickle                                                                   # CAMBIO

import plotly.graph_objects as go
import plotly.express as px

from sklearn.metrics import roc_auc_score                                       # CAMBIO

import lightgbm as lgb                                                          # CAMBIO

import optuna                                                                   # CAMBIO
from optuna.visualization import plot_optimization_history, plot_param_importances, plot_slice, plot_contour   # CAMBIO

In [ ]:
# Paths Variables

from pathlib import Path
nb_file = globals().get('__vsc_ipynb_file__')                 # ruta de la notebook (la define VS Code)
cwd = Path(nb_file).parent if nb_file else Path.cwd()         # carpeta de la notebook; si no es VS Code, usa el cwd
RAIZ = (cwd / '../../../../05_ExpMeses').resolve()

sys.path.insert(0, str(cwd))                                  # semillas.py está al lado de la notebook
from semillas import SEMILLAS as semillas

base_path = RAIZ.as_posix() + '/'                             # as_posix: barras normales también en Windows
print(RAIZ, RAIZ.exists(), (cwd / 'semillas.py').exists())

dataset_path = base_path + 'data/'
db_path = base_path + 'databases/'
modelos_path = base_path + 'modelos/'
out_path = base_path + 'out/'
dataset_file = 'competencia_02_meses.parquet'
ruta_meses = dataset_path + dataset_file
print(ruta_meses, Path(ruta_meses).exists())

os.makedirs(db_path, exist_ok=True)                            # CAMBIO: carpeta de las bases de Optuna
os.makedirs(modelos_path, exist_ok=True)                       # CAMBIO: carpeta de modelos

#### Carga de los Meses del Baseline
Se leen de `competencia_02_meses.parquet` los meses 201901 a 201910 y se dejan en una tabla en memoria de DuckDB (`d2019`). 201911 y 201912 son del dashboard y no se usan. Los meses 1 y 2 no se entrenan: solo hacen falta para calcular los lags.

In [ ]:
%%time
# CAMBIO: lee del parquet de meses; solo hasta 201910 (201911 y 201912 son del dashboard)
con = duckdb.connect()
con.execute(f"SET threads = {duckdb_threads}")
con.execute(f"SET memory_limit = '{duckdb_memory_limit}'")
con.execute("SET enable_progress_bar = false")

MES_DESDE, MES_HASTA = 201901, 201910

con.execute(f"""
    CREATE OR REPLACE TABLE d2019 AS
    SELECT * FROM read_parquet('{ruta_meses}')
    WHERE foto_mes BETWEEN {MES_DESDE} AND {MES_HASTA}
""")
print(con.execute("SELECT count(*) FROM d2019").fetchall()[0][0], 'filas')

#### Filas, Clientes, Variables y Clase Ternaria por Mes

In [ ]:
mes_resumen = con.sql("""
    SELECT foto_mes,
           count(*)                                      AS filas,
           count(DISTINCT numero_de_cliente)             AS clientes,
           count(*) - count(DISTINCT numero_de_cliente)  AS duplicados      -- tiene que dar 0
    FROM d2019
    GROUP BY foto_mes
    ORDER BY foto_mes
""").df()
mes_resumen

In [ ]:
import math

cols = [r[0] for r in con.execute("SELECT column_name FROM (DESCRIBE d2019)").fetchall()]
print(len(cols), 'columnas')
print('clase_ternaria' in cols)

n_col = 5                                          # cantidad de columnas a mostrar
n_fil = math.ceil(len(cols) / n_col)
relleno = cols + [''] * (n_fil * n_col - len(cols))     # completa con vacíos para que cierre el rectángulo
tabla = pd.DataFrame(np.array(relleno).reshape(n_col, n_fil).T)
tabla.style.hide(axis='index').hide(axis='columns')

In [ ]:
# Clase por mes. Esperado: 201911 solo con BAJA+1, 201912 sin clase
clase_mes = (con.sql("""
    SELECT foto_mes, coalesce(clase_ternaria, '(sin clase)') AS clase, count(*) AS filas
    FROM d2019
    GROUP BY 1, 2
    ORDER BY 1, 2
""").df().pivot(index='foto_mes', columns='clase', values='filas').fillna(0).astype(int))

total = clase_mes.sum(axis=1)
clase_mes['pct_BAJA+2'] = (clase_mes.get('BAJA+2', 0) / total * 100).round(3)
clase_mes

#### Tipos de Variables

In [ ]:
esquema = con.sql("DESCRIBE d2019").df()[['column_name', 'column_type']]
ids = ['numero_de_cliente', 'foto_mes']

tipos_num = ('TINYINT', 'SMALLINT', 'INTEGER', 'BIGINT', 'HUGEINT', 'UTINYINT', 'USMALLINT', 'UINTEGER', 'UBIGINT',
             'FLOAT', 'DOUBLE', 'DECIMAL')
es_num = esquema['column_type'].str.startswith(tipos_num)

cols_num = [c for c in esquema.loc[es_num, 'column_name'] if c not in ids]            # a las que se les mide nulos y ceros
cols_otras = [c for c in esquema.loc[~es_num, 'column_name'] if c not in ids]         # texto, fechas, etc.

print(len(esquema), 'columnas |', len(cols_num), 'numéricas |', len(cols_otras), 'no numéricas')
print('No numéricas:', cols_otras)           # clase_ternaria y, si aparecen, columnas 0/1 que llegaron como texto
esquema['column_type'].value_counts()

#### Feature Engineering -- Lags y Deltas 1 y 2

In [ ]:
%%time
ruta_fe = dataset_path + 'competencia_02_fe.parquet'        # CAMBIO: antes ruta_lags / competencia_02_lags.parquet

mes_idx = "(foto_mes // 100 * 12 + foto_mes % 100)"
cols_sql = ', '.join(f'"{c}"' for c in cols_num)

sel = []
for c in cols_num:
    sel.append(f'p1."{c}" AS "lag_1_{c}"')
    sel.append(f'p2."{c}" AS "lag_2_{c}"')
    sel.append(f'd."{c}" - p1."{c}" AS "delta_1_{c}"')
    sel.append(f'd."{c}" - p2."{c}" AS "delta_2_{c}"')

query = f"""
    WITH base AS (SELECT *, {mes_idx} AS mes_idx FROM d2019),
         prev AS (SELECT numero_de_cliente, mes_idx, {cols_sql} FROM base)
    SELECT d.* EXCLUDE (mes_idx), {', '.join(sel)}
    FROM base d
    LEFT JOIN prev p1 ON p1.numero_de_cliente = d.numero_de_cliente AND p1.mes_idx = d.mes_idx - 1
    LEFT JOIN prev p2 ON p2.numero_de_cliente = d.numero_de_cliente AND p2.mes_idx = d.mes_idx - 2
"""
con.execute(f"""
    COPY ({query} ORDER BY d.foto_mes, d.numero_de_cliente)
    TO '{ruta_fe}' (FORMAT PARQUET, COMPRESSION ZSTD)
""")
print('guardado:', ruta_fe, round(Path(ruta_fe).stat().st_size / 1e6), 'MB')

In [ ]:
# 1) filas, clientes y columnas (esperadas: las de d2019 + 4 por cada variable numérica)
n_fil, n_cli = con.execute(f"SELECT count(*), count(DISTINCT numero_de_cliente) FROM read_parquet('{ruta_fe}')").fetchall()[0]
n_col = len(con.execute(f"DESCRIBE SELECT * FROM read_parquet('{ruta_fe}')").fetchall())
print(n_fil, 'filas |', n_cli, 'clientes |', n_col, 'columnas | esperadas:', len(esquema) + 4 * len(cols_num))

# 2) % de nulos de los lags de una variable, por mes
con.execute(f"""
    SELECT foto_mes,
           round(100 * avg((lag_1_mcuentas_saldo IS NULL)::int), 2) AS pct_null_lag1,
           round(100 * avg((lag_2_mcuentas_saldo IS NULL)::int), 2) AS pct_null_lag2
    FROM read_parquet('{ruta_fe}') GROUP BY 1 ORDER BY 1
""").df()

##### Cargado de Datos para LightGBM

In [ ]:
%%time
# Carga de los meses del baseline desde el parquet de FE
meses_opt_train  = [201903, 201904, 201905, 201906]     # optimización: train
meses_opt_val    =  201908                              # optimización: validación
meses_test_train = [201905, 201906, 201907, 201908]     # reentrenamiento para el test
meses_test       =  201910                              # test

MESES = sorted(set(meses_opt_train + [meses_opt_val] + meses_test_train + [meses_test]))
lista_meses = ', '.join(str(m) for m in MESES)

data = con.execute(f"""
    SELECT * FROM read_parquet('{ruta_fe}')
    WHERE foto_mes IN ({lista_meses})
    ORDER BY foto_mes, numero_de_cliente
""").fetchdf()                                           # orden fijo de filas

print(data.shape)

# Filas por mes
print('\nFilas por mes:')
print(data['foto_mes'].value_counts().sort_index().to_string())

# Resumen de los bloques de meses, con sus filas
def filas(meses):
    meses = meses if isinstance(meses, list) else [meses]
    return int(data['foto_mes'].isin(meses).sum())

bloques = {
    'Optimización | train':      meses_opt_train,
    'Optimización | validación': meses_opt_val,
    'Test         | train':      meses_test_train,
    'Test         | test':       meses_test,
}
print('\nBloques de meses:')
for nombre, meses in bloques.items():
    print(f'{nombre:27s}: {meses}  ->  {filas(meses):,} filas')

##### Clase Binaria + X/y de Optimización (train con BAJA+1 y BAJA+2, validación con BAJA+2)

In [ ]:
# Clase binaria: BAJA+1 y BAJA+2 juntas (1) contra CONTINUA (0)
print('Filas sin clase:', int(data['clase_ternaria'].isna().sum()))      # tiene que dar 0 en estos meses
data['clase_binaria_b12'] = np.where(data['clase_ternaria'].isin(['BAJA+1', 'BAJA+2']), 1, 0)

# No entran a X: la clase, la binaria, el id y el mes
cols_descartar = ['clase_ternaria', 'clase_binaria_b12', 'numero_de_cliente', 'foto_mes']

# Optimización: train (meses 3 a 6) y validación (mes 8)
train_data = data[data['foto_mes'].isin(meses_opt_train)]
val_data   = data[data['foto_mes'] == meses_opt_val]

X_train = train_data.drop(columns=cols_descartar)
y_train = train_data['clase_binaria_b12']

X_val = val_data.drop(columns=cols_descartar)
y_val = (val_data['clase_ternaria'] == 'BAJA+2').astype(int)              # validación solo con BAJA+2

# feature_pre_filter=False: necesario porque Optuna va a variar min_data_in_leaf entre trials
train_data_lgb = lgb.Dataset(X_train, label=y_train, params={'feature_pre_filter': False})
val_data_lgb   = lgb.Dataset(X_val, label=y_val, reference=train_data_lgb)

print('train:', X_train.shape, '| positivos', int(y_train.sum()), f'({100 * y_train.mean():.2f}%)')
print('valid:', X_val.shape,   '| positivos BAJA+2', int(y_val.sum()),   f'({100 * y_val.mean():.2f}%)')

#### Optimización de Hiperparámetros (Optuna, validación en 201908)

In [ ]:
# Optimización bayesiana (Optuna): rangos del baseline.yaml, validación en el mes 201908
def objective(trial):

    params = {
        'objective': 'binary',
        'metric': 'auc',
        'boosting_type': 'gbdt',
        'boost_from_average': True,
        'feature_pre_filter': False,
      # bayesianos
        'num_leaves': trial.suggest_int('num_leaves', 8, 256),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 20, 5000, log=True),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.1, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.3, 1.0),
      # fijos
        'bagging_freq': 1,
        'seed': semillas[0],
        'num_threads': lgb_threads,
        'verbose': -1
    }

    model = lgb.train(
        params,
        train_data_lgb,
        num_boost_round=5000,                                           # techo; el corte lo pone el early stopping
        valid_sets=[val_data_lgb],                                      # validación en 201908 
        callbacks=[lgb.early_stopping(stopping_rounds=100, verbose=False)]
    )

    trial.set_user_attr('best_iter', model.best_iteration)              # iteraciones del mejor AUC
    return model.best_score['valid_0']['auc']                           # AUC en validación: ordena los trials

storage_name = "sqlite:///" + db_path + "ExpMeses_Optimization.db"
study_name = "EM_Baseline_TrB12_ValB2_T201903t06V201908"                # estudio nuevo (train fusionado, validación BAJA+2)

study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=semillas[0]),               # CAMBIO: sampler con semilla, reproducible
    study_name=study_name,
    storage=storage_name,
    load_if_exists=True,
)

print(len(study.trials))     # debe dar 0 si la base es nueva

In [ ]:
objetivo_total = 50                                        # número de trials que queremos correr en total
faltan = objetivo_total - len(study.trials)
print(f"Tengo {len(study.trials)}, corro {faltan} más")
if faltan > 0:
    study.optimize(objective, n_trials=faltan)

##### Resultados Optimización de Hiperparámetros

In [ ]:
# Historia de la optimización: AUC de cada trial y mejor AUC acumulado
optuna.visualization.plot_optimization_history(study)

In [ ]:
# Importancia de cada hiperparámetro: cuánto explica de las diferencias de AUC entre trials
plot_param_importances(study)

In [ ]:
# Slice: AUC de cada trial según el valor de cada hiperparámetro (uno por gráfico)
plot_slice(study)

In [ ]:
# Contour de un solo par: learning_rate contra min_data_in_leaf (el freno del baseline)
plot_contour(study, params=['learning_rate', 'min_data_in_leaf'])

##### Mejor Trial

In [ ]:
mejor = study.best_trial
print('Mejor trial:', mejor.number, '| AUC validación:', round(mejor.value, 4), '| best_iter:', mejor.user_attrs['best_iter'])
mejor.params

#### Train 201905-201908 y Test 201910 con 5 Semillas

##### Variables para Reentrenar y Testear

In [ ]:
# Test: train con los meses 5 a 8 y test en el mes 10
data_test_train = data[data['foto_mes'].isin(meses_test_train)]       # filas de 201905 a 201908 (incluye agosto, el mes de validación)
data_test       = data[data['foto_mes'] == meses_test]                # filas de 201910 (solo para medir el AUC final)

# X: variables sin la clase, el id y el mes (cols_descartar). y: clase binaria (BAJA+1 y BAJA+2 = 1, CONTINUA = 0)
X_test_train = data_test_train.drop(columns=cols_descartar)
y_test_train = data_test_train['clase_binaria_b12']
X_test       = data_test.drop(columns=cols_descartar)
y_test       = data_test['clase_binaria_b12']

# Dataset de LightGBM con el train del test; feature_pre_filter=False por consistencia con la optimización
data_test_train_lgb = lgb.Dataset(X_test_train, label=y_test_train, params={'feature_pre_filter': False})

# Chequeo: forma de cada matriz y cantidad de positivos
print('train:', X_test_train.shape, '| positivos', int(y_test_train.sum()), f'({100 * y_test_train.mean():.2f}%)')
print('test :', X_test.shape,       '| positivos', int(y_test.sum()),       f'({100 * y_test.mean():.2f}%)')

##### Reentrenamiento y Testeo con AUC y Ganancia

In [ ]:
# Parámetros de la ganancia
ganancia_acierto = 1_072_500                              # ganancia por cliente BAJA+2 enviado
costo_estimulo   = 27_500                                 # costo por cliente enviado que no es BAJA+2

# Ganancia con el BAJA+2 real: se acierta solo si el cliente fue BAJA+2, aunque el modelo haya aprendido las dos bajas
def ganancia_real(y_pred, y_true_class, umbral=0.025):
    ganancia = np.where(y_true_class == 'BAJA+2', ganancia_acierto, -costo_estimulo)
    return ganancia[y_pred >= umbral].sum()

In [ ]:
n_semillas_test = 5
best_iter = mejor.user_attrs['best_iter']                 # iteraciones del mejor trial (sin validación no hay early stopping)
y_test_class = data_test['clase_ternaria'].values         # clase real de octubre (BAJA+1, BAJA+2, CONTINUA), para la ganancia
y_test_baja2 = (y_test_class == 'BAJA+2').astype(int)     # 1 solo si el cliente es BAJA+2

rows = []
for seed in semillas[:n_semillas_test]:
    params = {
        'objective': 'binary', 'metric': 'auc', 'boosting_type': 'gbdt', 'boost_from_average': True,
        'feature_pre_filter': False,
        **mejor.params,
        'bagging_freq': 1,
        'seed': seed,
        'num_threads': lgb_threads,
        'verbose': -1
    }
    # Entrena con 201905-201908 y las iteraciones del mejor trial; guarda el modelo de cada semilla
    model = lgb.train(params, data_test_train_lgb, num_boost_round=best_iter)
    model.save_model(modelos_path + f'EM_Baseline_ValB2_Train0508_Test10_s{seed}.txt')
    pred = model.predict(X_test)                          # probabilidad de baja de cada cliente en octubre
    rows.append({
        'seed': seed,
        'auc_test': roc_auc_score(y_test_baja2, pred),    # AUC contra BAJA+2 solamente
        'ganancia': ganancia_real(pred, y_test_class),    # ganancia con umbral 0.025, solo BAJA+2 gana
    })

df_test = pd.DataFrame(rows)

In [ ]:
print(df_test)
print(f"AUC test 201910 (BAJA+2): {df_test['auc_test'].mean():.4f} ± {df_test['auc_test'].std():.4f}")
print(f"Ganancia test 201910    : {df_test['ganancia'].mean():,.0f} ± {df_test['ganancia'].std():,.0f}")

##### Curva de Ganancia Acumulada en Octubre (Baseline, una semilla)

In [ ]:
import plotly.graph_objects as go

# Ordena los clientes de octubre de mayor a menor probabilidad (modelo de la última semilla del loop)
orden = np.argsort(pred)[::-1]
# Resultado real de cada cliente en ese orden: gana solo si fue BAJA+2
ganancia_ordenada = np.where(y_test_baja2 == 1, ganancia_acierto, -costo_estimulo)[orden]
# Ganancia acumulada: lo que se gana si se estimulan los primeros 1, 2, 3... clientes
ganancia_acumulada = np.cumsum(ganancia_ordenada)

# Mejor corte: el N donde la ganancia acumulada llega a su máximo (+1 porque argmax empieza en 0)
mejor_corte = np.argmax(ganancia_acumulada) + 1
mejor_ganancia = ganancia_acumulada[mejor_corte - 1]
# Probabilidad del último cliente que entra en ese corte
p_corte = pred[orden][mejor_corte - 1]

limite = 20000  # cuántos clientes mostrar en el eje x

fig = go.Figure()

# Curva de ganancia acumulada, hasta el límite del eje
fig.add_trace(go.Scatter(
    x=list(range(1, limite + 1)), y=ganancia_acumulada[:limite],
    mode='lines', name='Ganancia acumulada', line=dict(color='blue')
))

# Línea vertical punteada en el corte óptimo
fig.add_vline(
    x=mejor_corte, line_dash='dash', line_color='black',
    annotation_text=f'Corte óptimo: {mejor_corte} clientes',
    annotation_position='top'
)

# Punto rojo en el máximo, con la probabilidad de corte al lado
fig.add_trace(go.Scatter(
    x=[mejor_corte], y=[mejor_ganancia],
    mode='markers+text', marker=dict(color='red', size=10),
    text=[f'p = {p_corte:.3f}'], textposition='middle right',
    showlegend=False
))

fig.update_layout(
    title=f'Ganancia acumulada en Octubre según clientes estimulados - Baseline, mejor trial {mejor.number}, semilla {seed}',
    xaxis_title='Cantidad de clientes ordenados por probabilidad descendente',
    yaxis_title='Ganancia acumulada',
    xaxis_range=[0, limite],
    template='plotly_white'
)

fig.show()

print(f"Máxima ganancia: {mejor_ganancia:.2f} en el corte {mejor_corte}, con probabilidad de corte p = {p_corte:.4f}")

##### Curva de Ganancia Acumulada en Octubre (Baseline, 5 semillas)

In [ ]:
import plotly.graph_objects as go

limite = 20000  # cuántos clientes mostrar en el eje x

fig = go.Figure()
for seed in semillas[:n_semillas_test]:
    # Carga el modelo guardado de esta semilla y predice octubre
    model_seed = lgb.Booster(model_file=modelos_path + f'EM_Baseline_ValB2_Train0508_Test10_s{seed}.txt')
    pred_seed = model_seed.predict(X_test)

    orden = np.argsort(pred_seed)[::-1]                   # clientes de octubre de mayor a menor probabilidad
    # Resultado real de cada cliente en ese orden: gana solo si fue BAJA+2
    ganancia_ordenada = np.where(y_test_baja2 == 1, ganancia_acierto, -costo_estimulo)[orden]
    ganancia_acumulada = np.cumsum(ganancia_ordenada)     # ganancia si se estimulan los primeros 1, 2, 3... clientes
    fig.add_trace(go.Scatter(
        x=list(range(1, limite + 1)), y=ganancia_acumulada[:limite],
        mode='lines', name=f'semilla {seed}'
    ))

fig.update_layout(
    title='Ganancia acumulada en Octubre según clientes estimulados - Baseline (5 semillas)',
    xaxis_title='Cantidad de clientes ordenados por probabilidad descendente',
    yaxis_title='Ganancia acumulada',
    xaxis_range=[0, limite],
    template='plotly_white'
)

fig.show()

##### Ganancia por Corte en Octubre (Tabla, 5 Semillas)

In [ ]:
cortes = np.arange(2000, 8001, 500)                       # cortes fijos (2.000, 2.500, ... 8.000) alrededor del máximo de la curva

rows_cortes = []                                          # ganancia de cada semilla en cada corte fijo
for seed in semillas[:n_semillas_test]:
    # Carga el modelo guardado de esta semilla y predice octubre
    model_seed = lgb.Booster(model_file=modelos_path + f'EM_Baseline_ValB2_Train0508_Test10_s{seed}.txt')
    pred_seed = model_seed.predict(X_test)

    orden = np.argsort(pred_seed)[::-1]                   # clientes de octubre de mayor a menor probabilidad
    # Resultado real de cada cliente en ese orden: gana solo si fue BAJA+2
    ganancia_ordenada = np.where(y_test_baja2 == 1, ganancia_acierto, -costo_estimulo)[orden]
    ganancia_acumulada = np.cumsum(ganancia_ordenada)     # ganancia si se estimulan los primeros 1, 2, 3... clientes

    # Ganancia de esta semilla en cada corte fijo (-1 porque la posición N está en el índice N-1)
    for corte in cortes:
        rows_cortes.append({
            "seed": seed,
            "corte": corte,
            "ganancia": ganancia_acumulada[corte - 1]
        })

df_ganancia_cortes = pd.DataFrame(rows_cortes)

# Estadísticas por corte sobre las 5 semillas
estadisticas_cortes = (
    df_ganancia_cortes.groupby('corte')['ganancia']
    .agg(['mean', 'std', 'min', 'max', 'median'])
)
estadisticas_cortes

##### Boxplot de Ganancia por Corte en Octubre (5 Semillas)

In [ ]:
import plotly.graph_objects as go

# Estadísticas por corte sobre las 5 semillas, más q1 y q3 (percentiles 25 y 75) para las cajas
estadisticas_cortes = (
    df_ganancia_cortes.groupby('corte')['ganancia']
    .agg(['mean', 'std', 'min', 'max', 'median',
          lambda x: x.quantile(0.25), lambda x: x.quantile(0.75)])
)
estadisticas_cortes.columns = ['mean', 'std', 'min', 'max', 'median', 'q1', 'q3']

fig = go.Figure()

# Un boxplot por corte, armado con las estadísticas ya calculadas
fig.add_trace(go.Box(
    x=[str(c) for c in estadisticas_cortes.index],         # corte como texto: cada uno es una caja separada
    q1=estadisticas_cortes['q1'],                          # borde inferior de la caja
    median=estadisticas_cortes['median'],                  # línea del medio
    q3=estadisticas_cortes['q3'],                          # borde superior de la caja
    lowerfence=estadisticas_cortes['min'],                 # bigote inferior: mínimo de las 5 semillas
    upperfence=estadisticas_cortes['max'],                 # bigote superior: máximo de las 5 semillas
    mean=estadisticas_cortes['mean'],                      # media (línea punteada)
    sd=estadisticas_cortes['std'],
    boxmean=True,
))

fig.update_layout(
    title=f'Distribución de ganancia por corte - Baseline ({n_semillas_test} semillas)',
    xaxis_title='Cantidad de clientes (corte)',
    yaxis_title='Ganancia acumulada',
    template='plotly_white'
)

fig.show()

##### Mejor Corte por Semilla y Mediana en Octubre

In [ ]:
# Mejor corte de cada semilla: el N donde la ganancia acumulada llega a su máximo
rows_N = []
for seed in semillas[:n_semillas_test]:
    # Carga el modelo guardado de esta semilla y predice octubre
    model_seed = lgb.Booster(model_file=modelos_path + f'EM_Baseline_ValB2_Train0508_Test10_s{seed}.txt')
    pred_seed = model_seed.predict(X_test)

    orden = np.argsort(pred_seed)[::-1]                   # clientes de octubre de mayor a menor probabilidad
    # Resultado real de cada cliente en ese orden: gana solo si fue BAJA+2
    ganancia_ordenada = np.where(y_test_baja2 == 1, ganancia_acierto, -costo_estimulo)[orden]
    ganancia_acumulada = np.cumsum(ganancia_ordenada)     # ganancia si se estimulan los primeros 1, 2, 3... clientes

    mejor_N = int(np.argmax(ganancia_acumulada)) + 1      # +1 porque argmax empieza en 0
    rows_N.append({"seed": seed, "mejor_N": mejor_N, "mejor_ganancia": ganancia_acumulada[mejor_N - 1]})

df_mejorN = pd.DataFrame(rows_N)
print(df_mejorN)

# N central: mediana de los mejores N (uno por semilla), y su peso como % de los clientes de octubre
clientes_octubre = len(y_test_baja2)
N_central = df_mejorN['mejor_N'].median()
pct_central = N_central / clientes_octubre

# Barrido: el mismo porcentaje central, corrido -20%, -10%, 0, +10% y +20%
barrido = pd.DataFrame({'variacion': [-0.20, -0.10, 0.0, 0.10, 0.20]})
barrido['pct'] = (pct_central * (1 + barrido['variacion'])).round(4)
barrido['N_octubre'] = (barrido['pct'] * clientes_octubre).round().astype(int)     # clientes que implica cada porcentaje en octubre

print(f"Mediana del mejor N en octubre: {N_central:.0f}  ->  {pct_central*100:.2f}% de {clientes_octubre} clientes")
barrido

#### Resultados del Baseline -- validación y testeo contra baja+2 únicamente --

